In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# timezone (UTC+8) per requirement
tz = pytz.timezone('Asia/Shanghai')

# Load telemetry CSVs
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')

# Convert timestamp to timezone-aware datetime in UTC+8 (Asia/Shanghai)
# Interpret the epoch as UTC then convert to Asia/Shanghai
df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Helper to compute aggregated table with required stats
def compute_aggregates(df, group_cols, value_col='value'):
    agg = (
        df.groupby(group_cols)[value_col]
        .agg(
            count='count',
            min='min',
            max='max',
            mean='mean',
            p50=lambda x: x.quantile(0.5),
            p95=lambda x: x.quantile(0.95),
        )
        .reset_index()
    )
    # Round numeric columns for compact display
    num_cols = ['min', 'max', 'mean', 'p50', 'p95']
    for c in num_cols:
        agg[c] = agg[c].astype(float).round(6)
    # sort for stable top results (by count desc, then max desc)
    agg = agg.sort_values(['count', 'max'], ascending=[False, False]).reset_index(drop=True)
    return agg

# Compute aggregates and keep full tables in memory
metric_agg_full = compute_aggregates(df_metric, ['cmdb_id', 'kpi_name'])
trace_agg_full = compute_aggregates(df_trace, ['cmdb_id', 'trace_name'])

# Prepare compact top-20 views for display
metric_agg_top = metric_agg_full.head(20)
trace_agg_top = trace_agg_full.head(20)

# Distinct lists (sorted) for metric
metric_cmdb_ids = pd.Series(sorted(df_metric['cmdb_id'].dropna().unique()), name='metric_cmdb_id')
metric_kpi_names = pd.Series(sorted(df_metric['kpi_name'].dropna().unique()), name='metric_kpi_name')

# Distinct lists (sorted) for trace
trace_cmdb_ids = pd.Series(sorted(df_trace['cmdb_id'].dropna().unique()), name='trace_cmdb_id')
trace_trace_names = pd.Series(sorted(df_trace['trace_name'].dropna().unique()), name='trace_name')

# Display compact outputs: top-20 aggregate tables and distinct lists
metric_agg_top, metric_cmdb_ids, metric_kpi_names, trace_agg_top, trace_cmdb_ids, trace_trace_names

```
Out[1]:
```
```python
summary = (
    "Summary of aggregates:\n"
    "- Metrics: grouped by (cmdb_id, kpi_name). Found 48 distinct cmdb_id values and 132 distinct kpi_name values.\n"
    "  Top examples (from top-20):\n"
    "    * osb_001.app.num: count=61, min=51.5, max=535.0, mean≈379.459, p50=415.0, p95=516.0\n"
    "    * osb_001.app.succee_num: count=61 (similar scale to app.num)\n"
    "    * db_001.CPU_free_pct: count=61, min≈98.099%, max≈99.452%, mean≈98.892%, p95≈99.279%\n"
    "    * Several redis keyspace metrics appear with large absolute values (count=60).\n"
    "  Note: many metric series have 61 samples (one per minute window in this dataset).\n"
    "- Traces: grouped by (cmdb_id, trace_name). Found 10 distinct cmdb_id values in trace data (docker_001..docker_008, os_021, os_022) and 76 distinct trace_name values.\n"
    "  Top examples (from top-20):\n"
    "    * docker_* . trace.self.jdbc.row_count: count=30 per docker, mean ≈1.56k, p50 in the ~1.2–1.9k range, p95 ~1.3–2.1k.\n"
    "    * trace.self.local.row_count and trace.self.csf.row_count show similar per-docker counts and mid-range medians.\n"
    "- Full aggregated tables are stored in memory as metric_agg_full and trace_agg_full; the displayed outputs were the top-20 rows (metric_agg_top, trace_agg_top).\n"
    "This is a concise overview of the aggregated statistics computed from metric.csv and trace.csv."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                  kpi_name  count           min           max          mean           p50           p95
0     osb_001                   app.num     61  5.150000e+01  5.350000e+02  3.794590e+02  4.150000e+02  5.160000e+02
1     osb_001            app.succee_num     61  0.000000e+00  5.350000e+02  3.509262e+02  4.120000e+02  5.160000e+02
2      db_001              CPU_free_pct     61  9.809940e+01  9.945272e+01  9.889217e+01  9.893286e+01  9.927915e+01
3      db_008              CPU_free_pct     61  9.679196e+01  9.868179e+01  9.805478e+01  9.802839e+01  9.862258e+01
4      db_007             MEM_real_util     61  8.209000e+01  8.214000e+01  8.211771e+01  8.212000e+01  8.214000e+01
5      os_012              Disk_io_util     61  0.000000e+00  1.000000e+00  2.147540e-01  0.000000e+00  5.246850e-01
6     osb_001           app.succee_rate     61  0.000000e+00  1.000000e+00  9.069130e-01  1.000000e+00  1.000000e+00
7      os_022  Outgoing_network_traffic     61  2.978000e-03  7.194100e-02  5.108600e-02  5.586600e-02  6.881200e-02
8     osb_001              app.avg_time     61  0.000000e+00  4.682700e-02  3.572000e-03  7.460000e-04  1.850300e-02
9      os_012      Processor_load_5_min     61  1.667000e-03  2.666700e-02  8.169000e-03  6.667000e-03  2.000000e-02
10     os_015  Incoming_network_traffic     61  0.000000e+00  3.460000e-03  2.199000e-03  2.294000e-03  2.369000e-03
11     os_008                Send_total     61  5.080000e-04  6.830000e-04  5.770000e-04  5.570000e-04  6.780000e-04
12     os_006                Send_total     61  3.040000e-04  4.740000e-04  3.680000e-04  3.660000e-04  4.710000e-04
13     os_007               Disk_rd_ios     61  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
14     os_020              Disk_io_util     61  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
15  redis_001             keyspace_hits     60  1.110128e+10  1.110812e+10  1.110462e+10  1.110461e+10  1.110777e+10
16  redis_002             keyspace_hits     60  1.100152e+10  1.100829e+10  1.100483e+10  1.100482e+10  1.100795e+10
17  redis_009             keyspace_hits     60  1.089034e+10  1.089704e+10  1.089361e+10  1.089360e+10  1.089670e+10
18  redis_002           keyspace_misses     60  8.595652e+09  8.601011e+09  8.598296e+09  8.598291e+09  8.600739e+09
19  redis_001           keyspace_misses     60  8.422388e+09  8.427641e+09  8.424977e+09  8.424972e+09  8.427375e+09, 0        db_001
1        db_002
2        db_003
3        db_004
4        db_005
        ...    
43    redis_008
44    redis_009
45    redis_010
46    redis_011
47    redis_012
Name: metric_cmdb_id, Length: 48, dtype: object, 0                   ACS
1                  AIOS
2                   AWS
3            Agent_ping
4           Asm_Free_Tb
             ...       
127        used_cpu_sys
128       used_cpu_user
129         used_memory
130    used_memory_peak
131     used_memory_rss
Name: metric_kpi_name, Length: 132, dtype: object,        cmdb_id                  trace_name  count    min     max         mean     p50      p95
0   docker_003   trace.self.jdbc.row_count     30  336.0  2387.0  1559.900000  1769.0  2096.00
1   docker_004   trace.self.jdbc.row_count     30  224.0  2353.0  1591.866667  1921.0  2234.85
2   docker_001   trace.self.jdbc.row_count     30  353.0  2260.0  1567.833333  1858.5  2120.50
3   docker_002   trace.self.jdbc.row_count     30  336.0  2209.0  1581.066667  1872.5  2201.35
4   docker_003  trace.self.local.row_count     30  294.0  2086.0  1364.066667  1547.0  1834.00
5   docker_004  trace.self.local.row_count     30  196.0  2058.0  1392.533333  1680.0  1955.10
6   docker_001  trace.self.local.row_count     30  308.0  1974.0  1370.133333  1624.0  1853.55
7   docker_002  trace.self.local.row_count     30  294.0  1932.0  1382.733333  1638.0  1925.70
8   docker_006   trace.self.jdbc.row_count     30  218.0  1412.0   990.566667  1171.5  1392.80
9   docker_008   trace.self.jdbc.row_count     30  207.0  1376.0   983.733333  1158.5  1344.25
10  docker_007   trace.self.jdbc.row_count     30  223.0  1360.0   985.266667  1177.5  1321.10
11  docker_005   trace.self.jdbc.row_count     30  214.0  1330.0   975.766667  1188.0  1299.45
12  docker_006  trace.self.local.row_count     30  130.0   850.0   595.100000   702.0   838.85
13  docker_008  trace.self.local.row_count     30  124.0   827.0   590.566667   692.5   803.65
14  docker_007  trace.self.local.row_count     30  134.0   818.0   590.833333   707.0   790.55
15  docker_005  trace.self.local.row_count     30  129.0   796.0   584.700000   711.0   775.95
16  docker_003    trace.self.csf.row_count     30   84.0   596.0   389.733333   442.0   524.00
17  docker_004    trace.self.csf.row_count     30   56.0   588.0   397.866667   480.0   558.60
18  docker_001    trace.self.csf.row_count     30   83.0   564.0   391.466667   464.0   523.30
19  docker_002    trace.self.csf.row_count     30   84.0   552.0   395.066667   468.0   550.20, 0    docker_001
1    docker_002
2    docker_003
3    docker_004
4    docker_005
5    docker_006
6    docker_007
7    docker_008
8        os_021
9        os_022
Name: trace_cmdb_id, dtype: object, 0     trace.from_docker_001.remoteprocess.duration_mean
1      trace.from_docker_001.remoteprocess.duration_p95
2        trace.from_docker_001.remoteprocess.error_rate
3         trace.from_docker_001.remoteprocess.row_count
4     trace.from_docker_002.remoteprocess.duration_mean
                            ...                        
71          trace.to_docker_007.remoteprocess.row_count
72      trace.to_docker_008.remoteprocess.duration_mean
73       trace.to_docker_008.remoteprocess.duration_p95
74         trace.to_docker_008.remoteprocess.error_rate
75          trace.to_docker_008.remoteprocess.row_count
Name: trace_name, Length: 76, dtype: object)```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# reuse existing df_metric and tz if present; otherwise load
try:
    df_metric
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')
    df_metric = pd.read_csv('metric.csv')
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure ts exists and is timezone-aware Asia/Shanghai
if 'ts' not in df_metric.columns:
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # enforce tz
    if df_metric['ts'].dt.tz is None:
        df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        df_metric['ts'] = df_metric['ts'].dt.tz_convert(tz)

# 1) Compute global p5 and p95 per (cmdb_id, kpi_name) using the full series (global thresholds)
thresholds = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(
        p5=lambda x: x.quantile(0.05),
        p95=lambda x: x.quantile(0.95)
    )
    .reset_index()
)

# Merge thresholds back to full metric table (keeps full intermediate table in memory)
df_metric_thresh = df_metric.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# 2) Filter to given UTC+8 time window: May 22, 2020 00:30 to 01:00 (inclusive)
start = pd.Timestamp('2020-05-22 00:30:00', tz=tz)
end = pd.Timestamp('2020-05-22 01:00:00', tz=tz)
df_window = df_metric_thresh[(df_metric_thresh['ts'] >= start) & (df_metric_thresh['ts'] <= end)].copy()

# 3) Mark anomalies based on global thresholds (p95 and p5 computed above)
# Priority: high if value >= p95, else low if value <= p5, else None
df_window['anomaly_type'] = np.where(
    df_window['value'] >= df_window['p95'], 'high',
    np.where(df_window['value'] <= df_window['p5'], 'low', None)
)

# Extract only anomaly rows
anomalies = df_window[df_window['anomaly_type'].notnull()].copy()

# Prepare per-cmdb_id summary: number_of_anomalies and earliest_anomaly_timestamp
if not anomalies.empty:
    per_cmdb_summary = (
        anomalies
        .groupby('cmdb_id')
        .agg(
            number_of_anomalies=('anomaly_type', 'count'),
            earliest_anomaly_timestamp=('ts', 'min')
        )
        .reset_index()
    )
    # format earliest timestamp as string in UTC+8 for compactness
    per_cmdb_summary['earliest_anomaly_timestamp'] = per_cmdb_summary['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')
    # Only keep components with anomalies (already done) and sort by count desc
    per_cmdb_summary = per_cmdb_summary.sort_values('number_of_anomalies', ascending=False).reset_index(drop=True)
else:
    per_cmdb_summary = pd.DataFrame(columns=['cmdb_id', 'number_of_anomalies', 'earliest_anomaly_timestamp'])

# Top 20 anomaly rows sorted by timestamp with required columns
if not anomalies.empty:
    top20_anomalies = (
        anomalies
        .sort_values('ts')
        .head(20)
        .loc[:, ['ts', 'cmdb_id', 'kpi_name', 'value', 'p5', 'p95', 'anomaly_type']]
        .copy()
    )
    # Format ts and round numeric thresholds/values for compact display
    top20_anomalies['ts'] = top20_anomalies['ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')
    top20_anomalies[['value','p5','p95']] = top20_anomalies[['value','p5','p95']].astype(float).round(6)
else:
    top20_anomalies = pd.DataFrame(columns=['ts','cmdb_id','kpi_name','value','p5','p95','anomaly_type'])

# Total anomaly count
total_anomalies = anomalies.shape[0]

# Display compact results (keep full intermediates like df_metric_thresh, thresholds, df_window in memory)
per_cmdb_summary, top20_anomalies, total_anomalies

```
Out[2]:
```
```python
summary = (
    "Summary of anomaly detection (window: 2020-05-22 00:30 to 01:00 UTC+8):\n"
    "- Method: global p5 and p95 were computed per (cmdb_id, kpi_name) from the full series. "
    "Rows in the specified time window were flagged 'high' if value >= p95 and 'low' if value <= p5.\n"
    "- Total anomalies found in the window: 21,135.\n"
    "- Number of components with anomalies: 48. Earliest anomaly timestamp (for components with anomalies): 2020-05-22 00:30:00 CST+0800.\n"
    "- Top components by anomaly count (top 5):\n"
    "    1) db_006: 692 anomalies\n"
    "    2) db_013: 654 anomalies\n"
    "    3) db_005: 653 anomalies\n"
    "    4) db_001: 634 anomalies\n"
    "    5) os_007: 622 anomalies\n"
    "  (Lowest in the returned list: osb_001 with 42 anomalies.)\n"
    "- Top anomaly rows (earliest timestamps) are dominated by entries at 2020-05-22 00:30:00 UTC+8, notably many metrics on os_019 (Disk_*, ICMP_ping, queue/process metrics) showing a mix of 'high' and 'low' flags.\n"
    "- Interpretation: anomalies are concentrated at the start of the window (00:30) across many components, with databases (db_*) and certain OS nodes (e.g., os_019, os_007) most affected. Investigate those components and the specific KPIs flagged 'high' or 'low' for root cause.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id  number_of_anomalies    earliest_anomaly_timestamp
0       db_006                  692  2020-05-22 00:30:00 CST+0800
1       db_013                  654  2020-05-22 00:30:00 CST+0800
2       db_005                  653  2020-05-22 00:30:00 CST+0800
3       db_001                  634  2020-05-22 00:30:00 CST+0800
4       os_007                  622  2020-05-22 00:30:00 CST+0800
..         ...                  ...                           ...
43  docker_004                  248  2020-05-22 00:30:00 CST+0800
44  docker_006                  242  2020-05-22 00:30:00 CST+0800
45  docker_001                  240  2020-05-22 00:30:00 CST+0800
46  docker_005                  238  2020-05-22 00:30:00 CST+0800
47     osb_001                   42  2020-05-22 00:30:00 CST+0800

[48 rows x 3 columns],                                  ts cmdb_id                   kpi_name        value            p5           p95 anomaly_type
37509  2020-05-22 00:30:00 CST+0800  db_001                        ACS      0.02000      0.009500      0.020000         high
38331  2020-05-22 00:30:00 CST+0800  os_019                 Agent_ping      1.00000      1.000000      1.000000         high
38338  2020-05-22 00:30:00 CST+0800  os_019                 Disk_await      0.00000      0.000000      1.000000          low
38339  2020-05-22 00:30:00 CST+0800  os_019               Disk_io_util      0.00000      0.000000      0.000000         high
38340  2020-05-22 00:30:00 CST+0800  os_019                Disk_rd_ios      0.00000      0.000000      0.000000         high
38341  2020-05-22 00:30:00 CST+0800  os_019                Disk_rd_kbs      0.00000      0.000000      0.000000         high
38342  2020-05-22 00:30:00 CST+0800  os_019                 Disk_svctm      1.50000      0.000000      1.000000         high
38343  2020-05-22 00:30:00 CST+0800  os_019                Disk_wr_ios      2.00000      1.000000      2.000000         high
38315  2020-05-22 00:30:00 CST+0800  os_018                  ICMP_ping      1.00000      1.000000      1.000000         high
38345  2020-05-22 00:30:00 CST+0800  os_019                  ICMP_ping      1.00000      1.000000      1.000000         high
38354  2020-05-22 00:30:00 CST+0800  os_019                    Page_pi      0.00000      0.000000      0.000000         high
38357  2020-05-22 00:30:00 CST+0800  os_019       Processor_load_1_min      0.00625      0.006646      0.040459          low
38359  2020-05-22 00:30:00 CST+0800  os_019    Received_errors_packets      0.00000      0.000000      0.000000         high
38361  2020-05-22 00:30:00 CST+0800  os_019             Received_queue      2.00000      0.000000      2.000000         high
38364  2020-05-22 00:30:00 CST+0800  os_019        Sent_errors_packets      0.00000      0.000000      0.000000         high
38366  2020-05-22 00:30:00 CST+0800  os_019                 Sent_queue      0.00000      0.000000      1.000000          low
38367  2020-05-22 00:30:00 CST+0800  os_019              Shared_memory  63904.00000  63904.000000  63908.000000          low
38352  2020-05-22 00:30:00 CST+0800  os_019   Num_of_running_processes      1.00000      1.000000      2.000000          low
38368  2020-05-22 00:30:00 CST+0800  os_019  System_block_queue_length      0.00000      0.000000      0.450000          low
38312  2020-05-22 00:30:00 CST+0800  os_018                 Disk_svctm      0.00000      0.000000      6.461500          low, 21135)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

# Reuse anomalies, tz, and other in-memory variables from prior steps.
# Compute normalized_severity per anomaly row (handle zero thresholds as absolute difference).

import numpy as np
import pandas as pd

# Ensure 'anomalies' exists
try:
    anomalies
except NameError:
    raise RuntimeError("anomalies DataFrame not found in memory. Run previous steps first.")

# Work on a copy to keep full intermediate in memory but avoid modifying original unexpectedly
anoms = anomalies.copy()

# Vectorized computation of normalized_severity
anoms['p5'] = anoms['p5'].astype(float)
anoms['p95'] = anoms['p95'].astype(float)
anoms['value'] = anoms['value'].astype(float)

sev = pd.Series(index=anoms.index, dtype=float)

# High anomalies
mask_high = anoms['anomaly_type'] == 'high'
high_p95_zero = mask_high & (anoms['p95'] == 0)
high_p95_nonzero = mask_high & (anoms['p95'] != 0)

sev.loc[high_p95_zero] = (anoms.loc[high_p95_zero, 'value'] - anoms.loc[high_p95_zero, 'p95']).abs()
sev.loc[high_p95_nonzero] = (
    (anoms.loc[high_p95_nonzero, 'value'] - anoms.loc[high_p95_nonzero, 'p95'])
    / anoms.loc[high_p95_nonzero, 'p95']
)

# Low anomalies
mask_low = anoms['anomaly_type'] == 'low'
low_p5_zero = mask_low & (anoms['p5'] == 0)
low_p5_nonzero = mask_low & (anoms['p5'] != 0)

sev.loc[low_p5_zero] = (anoms.loc[low_p5_zero, 'p5'] - anoms.loc[low_p5_zero, 'value']).abs()
sev.loc[low_p5_nonzero] = (
    (anoms.loc[low_p5_nonzero, 'p5'] - anoms.loc[low_p5_nonzero, 'value'])
    / anoms.loc[low_p5_nonzero, 'p5']
)

# If any remaining NaNs (e.g., anomaly_type unexpected), fill with 0
sev = sev.fillna(0.0)
anoms['normalized_severity'] = sev

# Aggregate per cmdb_id
severity_summary = (
    anoms
    .groupby('cmdb_id')
    .agg(
        anomaly_count=('anomaly_type', 'count'),
        earliest_anomaly_timestamp=('ts', 'min'),
        max_normalized_severity=('normalized_severity', 'max'),
        mean_normalized_severity=('normalized_severity', 'mean')
    )
    .reset_index()
)

# Format earliest timestamp and round numeric severity columns for compact display
severity_summary['earliest_anomaly_timestamp'] = severity_summary['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')
severity_summary['max_normalized_severity'] = severity_summary['max_normalized_severity'].round(6)
severity_summary['mean_normalized_severity'] = severity_summary['mean_normalized_severity'].round(6)

# Top 10 cmdb_id by max_normalized_severity
severity_top10 = severity_summary.sort_values('max_normalized_severity', ascending=False).head(10).reset_index(drop=True)

# For top 3 cmdb_id, return up to 10 representative anomaly rows each (sorted by timestamp)
top3_ids = severity_top10['cmdb_id'].head(3).tolist()

# Prepare example rows: sort by ts, then groupby cmdb_id and take head(10)
examples = anoms[anoms['cmdb_id'].isin(top3_ids)].sort_values(['cmdb_id', 'ts'])
examples_top_by_cmdb = examples.groupby('cmdb_id').head(10).copy()

# Select and format required columns
examples_top_by_cmdb = examples_top_by_cmdb.loc[:, ['ts','cmdb_id','kpi_name','value','p5','p95','anomaly_type','normalized_severity']]
examples_top_by_cmdb['ts'] = examples_top_by_cmdb['ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')
examples_top_by_cmdb[['value','p5','p95','normalized_severity']] = examples_top_by_cmdb[['value','p5','p95','normalized_severity']].round(6)

# Final compact outputs
severity_top10, examples_top_by_cmdb

```
Out[3]:
```
```python
summary = (
    "Severity summary (incident window 2020-05-22 00:30–01:00 UTC+8):\n\n"
    "Top 10 components by max_normalized_severity (columns: cmdb_id, anomaly_count, earliest_ts, max_norm_sev, mean_norm_sev):\n"
    " 1) db_009: 538 anomalies, earliest=2020-05-22 00:30:00, max_norm_sev=99.000000, mean_norm_sev=0.231889\n"
    " 2) db_002: 605 anomalies, earliest=2020-05-22 00:30:00, max_norm_sev=79.975610, mean_norm_sev=0.178082\n"
    " 3) os_007: 622 anomalies, earliest=2020-05-22 00:30:00, max_norm_sev=78.988002, mean_norm_sev=0.385514\n"
    " 4) db_011: 617 anomalies, earliest=2020-05-22 00:30:00, max_norm_sev=78.858156, mean_norm_sev=0.191033\n"
    " 5) db_004: 533 anomalies, earliest=2020-05-22 00:30:00, max_norm_sev=19.000000, mean_norm_sev=0.055643\n"
    " 6) os_012: 462 anomalies, earliest=2020-05-22 00:30:00, max_norm_sev=18.671735, mean_norm_sev=0.065841\n"
    " 7) os_018: 334 anomalies, earliest=2020-05-22 00:30:00, max_norm_sev=12.313096, mean_norm_sev=0.082853\n"
    " 8) db_003: 459 anomalies, earliest=2020-05-22 00:30:00, max_norm_sev=9.230754, mean_norm_sev=0.104655\n"
    " 9) db_010: 571 anomalies, earliest=2020-05-22 00:30:00, max_norm_sev=8.718052, mean_norm_sev=0.061187\n"
    "10) os_022: 440 anomalies, earliest=2020-05-22 00:30:00, max_norm_sev=8.288281, mean_norm_sev=0.054706\n\n"
    "Representative anomaly rows were returned for the top 3 components (db_009, db_002, os_007) — up to 10 rows each, sorted by timestamp. "
    "Note: many representative rows show normalized_severity == 0.0 where value equals the threshold or the threshold is zero; nevertheless the aggregated max_normalized_severity shows there are individual very large deviations (see max_norm_sev above).\n\n"
    "Actionable insight: prioritize investigation on db_009, db_002, os_007 and db_011 first due to highest peak severities, then check other db_* and OS nodes in the top-10 list. Representative anomaly rows for the top-3 components are available in memory for drilling down."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(  cmdb_id  anomaly_count    earliest_anomaly_timestamp  max_normalized_severity  mean_normalized_severity
0  db_009            538  2020-05-22 00:30:00 CST+0800                99.000000                  0.231889
1  db_002            605  2020-05-22 00:30:00 CST+0800                79.975610                  0.178082
2  os_007            622  2020-05-22 00:30:00 CST+0800                78.988002                  0.385514
3  db_011            617  2020-05-22 00:30:00 CST+0800                78.858156                  0.191033
4  db_004            533  2020-05-22 00:30:00 CST+0800                19.000000                  0.055643
5  os_012            462  2020-05-22 00:30:00 CST+0800                18.671735                  0.065841
6  os_018            334  2020-05-22 00:30:00 CST+0800                12.313096                  0.082853
7  db_003            459  2020-05-22 00:30:00 CST+0800                 9.230754                  0.104655
8  db_010            571  2020-05-22 00:30:00 CST+0800                 8.718052                  0.061187
9  os_022            440  2020-05-22 00:30:00 CST+0800                 8.288281                  0.054706,                                  ts cmdb_id       kpi_name     value        p5       p95 anomaly_type  normalized_severity
37541  2020-05-22 00:30:00 CST+0800  db_002            ACS  0.010000  0.010000  0.020000          low              0.00000
37542  2020-05-22 00:30:00 CST+0800  db_002           AIOS  0.000000  0.000000  0.000000         high              0.00000
37543  2020-05-22 00:30:00 CST+0800  db_002            AWS  0.020000  0.000000  0.020000         high              0.00000
37550  2020-05-22 00:30:00 CST+0800  db_002           Hang  1.000000  1.000000  1.000000         high              0.00000
37554  2020-05-22 00:30:00 CST+0800  db_002  Login_Per_Sec  0.400000  0.400000  0.500000          low              0.00000
...                             ...     ...            ...       ...       ...       ...          ...                  ...
38044  2020-05-22 00:30:00 CST+0800  os_007    Disk_rd_kbs  0.000000  0.000000  0.000000         high              0.00000
38045  2020-05-22 00:30:00 CST+0800  os_007     Disk_svctm  0.000000  0.000000  0.075000          low              0.00000
38046  2020-05-22 00:30:00 CST+0800  os_007    Disk_wr_ios  0.000000  0.000000  0.000000         high              0.00000
38047  2020-05-22 00:30:00 CST+0800  os_007    Disk_wr_kbs  0.004273  0.002473  0.004153         high              0.02882
38048  2020-05-22 00:30:00 CST+0800  os_007      ICMP_ping  1.000000  1.000000  1.000000         high              0.00000

[30 rows x 8 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Reuse in-memory 'anoms' (anomalies with normalized_severity) if available
try:
    anoms
except NameError:
    try:
        anomalies  # older name
        anoms = anomalies.copy()
    except NameError:
        raise RuntimeError("Anomalies DataFrame not found in memory. Run prior steps first.")

# Filter for db_009
db009 = anoms[anoms['cmdb_id'] == 'db_009'].copy()

# If no anomalies for db_009, return empty results
if db009.empty:
    kpi_top1 = pd.DataFrame(columns=['kpi_name','max_normalized_severity','earliest_anomaly_timestamp'])
    db009_examples = pd.DataFrame(columns=['ts','cmdb_id','kpi_name','value','p5','p95','anomaly_type','normalized_severity'])
else:
    # Aggregate per kpi_name: max normalized_severity and earliest anomaly timestamp
    kpi_summary = (
        db009
        .groupby('kpi_name')
        .agg(
            max_normalized_severity=('normalized_severity', 'max'),
            earliest_anomaly_timestamp=('ts', 'min')
        )
        .reset_index()
    )
    # Format timestamp and round severity
    kpi_summary['earliest_anomaly_timestamp'] = kpi_summary['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')
    kpi_summary['max_normalized_severity'] = kpi_summary['max_normalized_severity'].round(6)

    # KPI with maximum normalized severity
    kpi_top1 = kpi_summary.sort_values('max_normalized_severity', ascending=False).head(1).reset_index(drop=True)

    # Prepare up to 20 representative anomaly rows for db_009 sorted by timestamp (earliest first)
    db009_examples = (
        db009
        .sort_values('ts')
        .head(20)
        .loc[:, ['ts','cmdb_id','kpi_name','value','p5','p95','anomaly_type','normalized_severity']]
        .copy()
    )
    db009_examples['ts'] = db009_examples['ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')
    db009_examples[['value','p5','p95','normalized_severity']] = db009_examples[['value','p5','p95','normalized_severity']].astype(float).round(6)

# Return compact outputs: top KPI info and example rows
kpi_top1, db009_examples

```
Out[4]:
```
```python
summary = (
    "db_009 anomaly summary:\n\n"
    "A) KPI with maximum normalized severity:\n"
    "   - kpi_name: SeqRead_Per_Sec\n"
    "   - max_normalized_severity: 99.0\n"
    "   - earliest anomaly timestamp for this KPI: 2020-05-22 00:31:00 CST+0800\n\n"
    "B) Representative anomaly rows:\n"
    "   - Up to 20 anomaly rows for db_009 were returned, sorted earliest-first (window 2020-05-22 00:30–01:00 UTC+8).\n"
    "   - Many displayed rows occur at 2020-05-22 00:30:00 or 00:31:00 and show several KPIs with normalized_severity == 0.0 (this happens when value equals p5/p95 or threshold is zero).\n"
    "   - Despite many zero-severity examples, SeqRead_Per_Sec shows a very large deviation (max severity 99.0) and should be prioritized for investigation.\n\n"
    "Use the returned representative rows (in memory) to drill down on exact values and timestamps for db_009."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(          kpi_name  max_normalized_severity    earliest_anomaly_timestamp
0  SeqRead_Per_Sec                     99.0  2020-05-22 00:31:00 CST+0800,                                  ts cmdb_id               kpi_name      value         p5         p95 anomaly_type  normalized_severity
37772  2020-05-22 00:30:00 CST+0800  db_009                   AIOS    0.00000    0.00000     0.00000         high                  0.0
37802  2020-05-22 00:30:00 CST+0800  db_009    tnsping_result_time    0.00000    0.00000    10.00000          low                  0.0
37801  2020-05-22 00:30:00 CST+0800  db_009            User_Commit    0.63000    0.63000     0.66100          low                  0.0
37795  2020-05-22 00:30:00 CST+0800  db_009        SctRead_Per_Sec    0.00000    0.00000     0.03000          low                  0.0
37794  2020-05-22 00:30:00 CST+0800  db_009               Row_Lock    0.00000    0.00000     0.00000         high                  0.0
37790  2020-05-22 00:30:00 CST+0800  db_009  Physical_Read_Per_Sec    0.00000    0.00000  2839.10500          low                  0.0
37797  2020-05-22 00:30:00 CST+0800  db_009            Sess_Active    1.90000    1.90000     2.00000          low                  0.0
37785  2020-05-22 00:30:00 CST+0800  db_009              MEM_Total  254.61033  254.61033   254.61033         high                  0.0
37784  2020-05-22 00:30:00 CST+0800  db_009          Login_Per_Sec    0.40000    0.40000     0.60000          low                  0.0
37780  2020-05-22 00:30:00 CST+0800  db_009                   Hang    1.00000    1.00000     1.00000         high                  0.0
37777  2020-05-22 00:30:00 CST+0800  db_009    DFParaWrite_Per_Sec    0.00000    0.00000     0.00000         high                  0.0
37773  2020-05-22 00:30:00 CST+0800  db_009                    AWS    0.02000    0.00000     0.02000         high                  0.0
37789  2020-05-22 00:30:00 CST+0800  db_009           On_Off_State    1.00000    1.00000     1.00000         high                  0.0
39061  2020-05-22 00:31:00 CST+0800  db_009        SctRead_Per_Sec    0.00000    0.00000     0.03000          low                  0.0
39071  2020-05-22 00:31:00 CST+0800  db_009    tnsping_result_time    0.00000    0.00000    10.00000          low                  0.0
39070  2020-05-22 00:31:00 CST+0800  db_009            User_Commit    0.63000    0.63000     0.66100          low                  0.0
39069  2020-05-22 00:31:00 CST+0800  db_009            TempTbs_Pct    0.06000    0.06000     0.06000         high                  0.0
39066  2020-05-22 00:31:00 CST+0800  db_009         Sess_Used_Undo    0.00000    0.00000     0.00000         high                  0.0
39065  2020-05-22 00:31:00 CST+0800  db_009         Sess_Used_Temp   13.00000   13.00000    13.00000         high                  0.0
39063  2020-05-22 00:31:00 CST+0800  db_009            Sess_Active    1.90000    1.90000     2.00000          low                  0.0)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# Reuse in-memory anomalies DataFrame 'anoms' and timezone 'tz' if available; otherwise try to recover from prior variables
try:
    anoms
except NameError:
    try:
        anomalies
        anoms = anomalies.copy()
    except NameError:
        raise RuntimeError("Anomalies DataFrame not found in memory. Run prior steps first.")

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Incident window (UTC+8)
start = pd.Timestamp('2020-05-22 00:30:00', tz=tz)
end = pd.Timestamp('2020-05-22 01:00:00', tz=tz)

# Filter for db_009 and SeqRead_Per_Sec within the window
db009_seq = anoms[
    (anoms['cmdb_id'] == 'db_009') &
    (anoms['kpi_name'] == 'SeqRead_Per_Sec') &
    (anoms['ts'] >= start) &
    (anoms['ts'] <= end)
].copy()

# Count of anomaly rows for this KPI in the window
db009_seq_count = db009_seq.shape[0]

# Prepare up to 20 rows sorted by timestamp ascending and format for compact display
db009_seq_out = (
    db009_seq
    .sort_values('ts')
    .head(20)
    .loc[:, ['ts','cmdb_id','kpi_name','value','p5','p95','anomaly_type','normalized_severity']]
    .copy()
)

# Format timestamp and numeric columns
db009_seq_out['ts'] = db009_seq_out['ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')
db009_seq_out[['value','p5','p95','normalized_severity']] = db009_seq_out[['value','p5','p95','normalized_severity']].astype(float).round(6)

# Return compact outputs: up to 20 rows and the count
db009_seq_out, db009_seq_count

```
Out[5]:
```
```python
summary = (
    "Filtered anomalies for db_009 / SeqRead_Per_Sec (window 2020-05-22 00:30–01:00 UTC+8):\n"
    "- Total anomaly rows found: 14\n"
    "- p5 = 0.0, p95 = 0.0105\n"
    "- Time range (anomalies): earliest 2020-05-22 00:31:00, latest 2020-05-22 00:59:00\n"
    "- Most rows (12 of 14) are 'low' with value = 0.00; these show normalized_severity = 0.0 because p5 == 0 and value == p5.\n"
    "- Notable 'high' breaches:\n"
    "    * 2020-05-22 00:52:00 — value = 1.05, anomaly_type = high, normalized_severity = 99.0 (largest deviation)\n"
    "    * 2020-05-22 00:56:00 — value = 0.02, anomaly_type = high, normalized_severity ≈ 0.904762\n"
    "- Recommendation: prioritize investigation of the 00:52:00 breach (very large deviation), then the 00:56:00 breach. The repeated zero values may reflect expected idle/zero readings (they equal p5), so focus on the nonzero high spikes for root-cause analysis."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                 ts cmdb_id         kpi_name  value   p5     p95 anomaly_type  normalized_severity
39062  2020-05-22 00:31:00 CST+0800  db_009  SeqRead_Per_Sec   0.00  0.0  0.0105          low             0.000000
44082  2020-05-22 00:35:00 CST+0800  db_009  SeqRead_Per_Sec   0.00  0.0  0.0105          low             0.000000
46597  2020-05-22 00:37:00 CST+0800  db_009  SeqRead_Per_Sec   0.00  0.0  0.0105          low             0.000000
49054  2020-05-22 00:39:00 CST+0800  db_009  SeqRead_Per_Sec   0.00  0.0  0.0105          low             0.000000
51565  2020-05-22 00:41:00 CST+0800  db_009  SeqRead_Per_Sec   0.00  0.0  0.0105          low             0.000000
56577  2020-05-22 00:45:00 CST+0800  db_009  SeqRead_Per_Sec   0.00  0.0  0.0105          low             0.000000
59106  2020-05-22 00:47:00 CST+0800  db_009  SeqRead_Per_Sec   0.00  0.0  0.0105          low             0.000000
61629  2020-05-22 00:49:00 CST+0800  db_009  SeqRead_Per_Sec   0.00  0.0  0.0105          low             0.000000
64088  2020-05-22 00:51:00 CST+0800  db_009  SeqRead_Per_Sec   0.00  0.0  0.0105          low             0.000000
65298  2020-05-22 00:52:00 CST+0800  db_009  SeqRead_Per_Sec   1.05  0.0  0.0105         high            99.000000
69067  2020-05-22 00:55:00 CST+0800  db_009  SeqRead_Per_Sec   0.00  0.0  0.0105          low             0.000000
70322  2020-05-22 00:56:00 CST+0800  db_009  SeqRead_Per_Sec   0.02  0.0  0.0105         high             0.904762
71620  2020-05-22 00:57:00 CST+0800  db_009  SeqRead_Per_Sec   0.00  0.0  0.0105          low             0.000000
74160  2020-05-22 00:59:00 CST+0800  db_009  SeqRead_Per_Sec   0.00  0.0  0.0105          low             0.000000, 14)```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# Reuse in-memory 'anoms' (anomaly rows within the incident window) if available
try:
    anoms
except NameError:
    try:
        anomalies
        anoms = anomalies.copy()
    except NameError:
        raise RuntimeError("Anomalies DataFrame not found in memory. Run prior steps first.")

# Ensure timezone variable
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Select top 5 anomalies by normalized_severity (descending)
cols = ['ts','cmdb_id','kpi_name','value','p5','p95','anomaly_type','normalized_severity']
top5_anomalies = (
    anoms
    .sort_values('normalized_severity', ascending=False)
    .head(5)
    .loc[:, cols]
    .copy()
)

# Format timestamp and numeric columns for compact display
if not top5_anomalies.empty:
    top5_anomalies['ts'] = top5_anomalies['ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')
    top5_anomalies[['value','p5','p95','normalized_severity']] = top5_anomalies[['value','p5','p95','normalized_severity']].astype(float).round(6)

top5_anomalies

```
Out[6]:
```
```python
summary = (
    "Top 5 anomalies by normalized_severity (window 2020-05-22 00:30–01:00 UTC+8):\n"
    "1) 2020-05-22 00:52:00 CST+0800 — db_009 — SeqRead_Per_Sec: value=1.05, p5=0.00, p95=0.0105, type=high, normalized_severity=99.0\n"
    "2) 2020-05-22 00:58:00 CST+0800 — db_002 — Physical_Read_Per_Sec: value=16.6, p5=0.00, p95=0.2050, type=high, normalized_severity≈79.97561\n"
    "3) 2020-05-22 00:44:00 CST+0800 — os_007 — Processor_load_1_min: value=0.006667, p5=0.00, p95=0.000083, type=high, normalized_severity≈78.98800\n"
    "4) 2020-05-22 00:33:00 CST+0800 — db_011 — LFParaWrite_Per_Sec: value=16.89, p5=0.05, p95=0.2115, type=high, normalized_severity≈78.85816\n"
    "5) 2020-05-22 00:48:00 CST+0800 — os_007 — Disk_svctm: value=3.5, p5=0.00, p95=0.0750, type=high, normalized_severity≈45.66667\n\n"
    "All five are 'high' breaches and represent the largest relative deviations from their global thresholds. "
    "Recommendation: prioritize investigation of these specific timestamps and KPIs (start with the db_009 SeqRead_Per_Sec spike) as they indicate the most severe anomalies in the incident window."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

ts cmdb_id               kpi_name      value    p5       p95 anomaly_type  normalized_severity
65298  2020-05-22 00:52:00 CST+0800  db_009        SeqRead_Per_Sec   1.050000  0.00  0.010500         high            99.000000
72662  2020-05-22 00:58:00 CST+0800  db_002  Physical_Read_Per_Sec  16.600000  0.00  0.205000         high            79.975610
55589  2020-05-22 00:44:00 CST+0800  os_007   Processor_load_1_min   0.006667  0.00  0.000083         high            78.988002
41633  2020-05-22 00:33:00 CST+0800  db_011    LFParaWrite_Per_Sec  16.890000  0.05  0.211500         high            78.858156
60625  2020-05-22 00:48:00 CST+0800  os_007             Disk_svctm   3.500000  0.00  0.075000         high            45.666667```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# Reuse in-memory anomalies DataFrame 'anoms' and timezone 'tz'
try:
    anoms
except NameError:
    try:
        anomalies
        anoms = anomalies.copy()
    except NameError:
        raise RuntimeError("Anomalies DataFrame not found in memory. Run prior steps first.")

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure ts is datetime with correct tz
if anoms['ts'].dt.tz is None:
    anoms['ts'] = pd.to_datetime(anoms['ts']).dt.tz_localize(tz)
else:
    anoms['ts'] = anoms['ts'].dt.tz_convert(tz)

# Candidate pairs
candidates = [
    ('db_009', 'SeqRead_Per_Sec'),
    ('db_002', 'Physical_Read_Per_Sec'),
    ('os_007', 'Processor_load_1_min'),
    ('db_011', 'LFParaWrite_Per_Sec'),
    ('os_007', 'Disk_svctm'),
]

rows = []
for cmdb_id, kpi in candidates:
    df = anoms[(anoms['cmdb_id'] == cmdb_id) & (anoms['kpi_name'] == kpi)].copy()
    total_anomaly_count = len(df)
    high_df = df[df['anomaly_type'] == 'high'].sort_values('ts').copy()
    number_of_high_anomaly_points = len(high_df)
    if number_of_high_anomaly_points == 0:
        longest_len = 0
        earliest_of_longest = None
        timestamps_of_longest = ""
    else:
        # compute minute gaps
        delta_min = high_df['ts'].diff().dt.total_seconds().div(60).fillna(9999)
        # new group when gap != 1 minute
        grp = (delta_min != 1).cumsum()
        high_df = high_df.assign(run_group=grp)
        group_stats = (
            high_df.groupby('run_group')
            .agg(start_ts=('ts', 'min'), length=('ts', 'count'), timestamps=('ts', lambda x: list(x)))
            .reset_index(drop=True)
        )
        # find longest run (if tie, pick earliest start_ts)
        max_len = group_stats['length'].max()
        candidates_runs = group_stats[group_stats['length'] == max_len].sort_values('start_ts')
        chosen = candidates_runs.iloc[0]
        longest_len = int(chosen['length'])
        earliest_of_longest = chosen['start_ts'].strftime('%Y-%m-%d %H:%M:%S %Z%z')
        # format up to 10 timestamps
        ts_list = [t.strftime('%Y-%m-%d %H:%M:%S %Z%z') for t in chosen['timestamps']]
        timestamps_of_longest = '; '.join(ts_list[:10])
    rows.append({
        'cmdb_id': cmdb_id,
        'kpi_name': kpi,
        'total_anomaly_count_in_window': int(total_anomaly_count),
        'number_of_high_anomaly_points': int(number_of_high_anomaly_points),
        'longest_high_run_length_minutes': int(longest_len),
        'earliest_timestamp_of_longest_high_run': earliest_of_longest,
        'timestamps_of_longest_run_sample': timestamps_of_longest
    })

summary_df = pd.DataFrame(rows)

# Keep final output compact: only the 5 summary rows as requested
summary_df

```
Out[7]:
```
Summary of consecutive-minute high-anomaly runs for the five candidate KPI pairs (window 2020-05-22 00:30–01:00 UTC+8):

1) db_009 — SeqRead_Per_Sec
- total anomaly rows in window: 14
- number of 'high' anomaly points: 2
- longest consecutive-high run: 1 minute
- earliest start of longest run: 2020-05-22 00:52:00 CST+0800
- timestamps of that run (up to 10): 2020-05-22 00:52:00 CST+0800

2) db_002 — Physical_Read_Per_Sec
- total anomaly rows in window: 27
- number of 'high' anomaly points: 2
- longest consecutive-high run: 1 minute
- earliest start of longest run: 2020-05-22 00:48:00 CST+0800
- timestamps of that run: 2020-05-22 00:48:00 CST+0800

3) os_007 — Processor_load_1_min
- total anomaly rows in window: 30
- number of 'high' anomaly points: 3
- longest consecutive-high run: 3 minutes
- earliest start of longest run: 2020-05-22 00:44:00 CST+0800
- timestamps of that longest run (up to 10): 2020-05-22 00:44:00 CST+0800; 2020-05-22 00:45:00 CST+0800; 2020-05-22 00:46:00 CST+0800

4) db_011 — LFParaWrite_Per_Sec
- total anomaly rows in window: 5
- number of 'high' anomaly points: 1
- longest consecutive-high run: 1 minute
- earliest start of longest run: 2020-05-22 00:33:00 CST+0800
- timestamps of that run: 2020-05-22 00:33:00 CST+0800

5) os_007 — Disk_svctm
- total anomaly rows in window: 30
- number of 'high' anomaly points: 3
- longest consecutive-high run: 1 minute
- earliest start of longest run: 2020-05-22 00:43:00 CST+0800
- timestamps of that run: 2020-05-22 00:43:00 CST+0800

Interpretation / next step:
- The most notable sustained high run is os_007 → Processor_load_1_min (3 consecutive minutes starting 00:44). The other candidates show high spikes but only single-minute consecutive runs. Prioritize investigating os_007 (processor load) and the large-severity single-minute spikes (e.g., db_009 at 00:52) for root-cause analysis.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id               kpi_name  total_anomaly_count_in_window  number_of_high_anomaly_points  longest_high_run_length_minutes earliest_timestamp_of_longest_high_run                   timestamps_of_longest_run_sample
0  db_009        SeqRead_Per_Sec                             14                              2                                1           2020-05-22 00:52:00 CST+0800                       2020-05-22 00:52:00 CST+0800
1  db_002  Physical_Read_Per_Sec                             27                              2                                1           2020-05-22 00:48:00 CST+0800                       2020-05-22 00:48:00 CST+0800
2  os_007   Processor_load_1_min                             30                              3                                3           2020-05-22 00:44:00 CST+0800  2020-05-22 00:44:00 CST+0800; 2020-05-22 00:45...
3  db_011    LFParaWrite_Per_Sec                              5                              1                                1           2020-05-22 00:33:00 CST+0800                       2020-05-22 00:33:00 CST+0800
4  os_007             Disk_svctm                             30                              3                                1           2020-05-22 00:43:00 CST+0800                       2020-05-22 00:43:00 CST+0800```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz
import numpy as np

# Reuse in-memory anomaly table 'anoms' (anomalies within the incident window)
try:
    anoms
except NameError:
    try:
        anomalies
        anoms = anomalies.copy()
    except NameError:
        raise RuntimeError("Anomalies DataFrame not found in memory. Run prior steps first.")

# Ensure timezone is Asia/Shanghai
tz = pytz.timezone('Asia/Shanghai')
if anoms['ts'].dt.tz is None:
    anoms['ts'] = anoms['ts'].dt.tz_localize(tz)
else:
    anoms['ts'] = anoms['ts'].dt.tz_convert(tz)

# Filter only 'high' anomaly points since we are summarizing high runs
highs = anoms[anoms['anomaly_type'] == 'high'].copy()

# If no high anomalies, return empty summary
if highs.empty:
    per_cmdb_summary = pd.DataFrame(columns=[
        'cmdb_id',
        'max_longest_high_run_length',
        'earliest_start_of_max_run',
        'max_normalized_severity_among_high_points',
        'number_of_kpis_with_high_runs'
    ])
else:
    # Compute consecutive runs per (cmdb_id, kpi_name)
    highs = highs.sort_values(['cmdb_id','kpi_name','ts']).reset_index(drop=True)

    # Compute minute gaps within each (cmdb_id,kpi_name) group
    highs['prev_ts'] = highs.groupby(['cmdb_id','kpi_name'])['ts'].shift(1)
    highs['gap_min'] = (highs['ts'] - highs['prev_ts']).dt.total_seconds().div(60)
    # start a new run where gap != 1 minute or first row (NaN)
    highs['new_run'] = ((highs['gap_min'] != 1) | highs['gap_min'].isna()).astype(int)
    highs['run_id'] = highs.groupby(['cmdb_id','kpi_name'])['new_run'].cumsum()

    # Aggregate runs to get length and start timestamp per run
    runs = (
        highs
        .groupby(['cmdb_id','kpi_name','run_id'])
        .agg(
            run_start_ts=('ts','min'),
            run_length=('ts','count')
        )
        .reset_index()
    )

    # For each (cmdb_id,kpi_name) find the longest run and its earliest start (if tie, earliest start)
    pair_longest = (
        runs
        .sort_values(['cmdb_id','kpi_name','run_length','run_start_ts'], ascending=[True,True,False,True])
        .groupby(['cmdb_id','kpi_name'], as_index=False)
        .first()[['cmdb_id','kpi_name','run_length','run_start_ts']]
        .rename(columns={'run_length':'longest_run_length','run_start_ts':'earliest_start_of_longest_run'})
    )

    # Also compute max normalized_severity among high points per pair (cmdb_id,kpi_name)
    pair_sev = (
        highs
        .groupby(['cmdb_id','kpi_name'], as_index=False)
        .agg(max_norm_sev_pair=('normalized_severity','max'))
    )

    pair_longest = pair_longest.merge(pair_sev, on=['cmdb_id','kpi_name'], how='left')

    # Now aggregate per cmdb_id:
    # - max_longest_high_run_length (minutes)
    # - earliest_start_of_max_run (timestamp) -> among pairs that achieve the max length, take earliest start
    # - max_normalized_severity_among_high_points -> max of normalized_severity across highs for that cmdb_id
    # - number_of_kpis_with_high_runs -> count of kpis with longest_run_length > 0
    per_cmdb_maxlen = (
        pair_longest
        .groupby('cmdb_id', as_index=False)
        .agg(
            max_longest_high_run_length=('longest_run_length','max')
        )
    )

    # earliest start among pairs that have the max length per cmdb_id
    merged = pair_longest.merge(per_cmdb_maxlen, on='cmdb_id', how='left')
    candidates_for_earliest = merged[merged['longest_run_length'] == merged['max_longest_high_run_length']].copy()
    earliest_starts = (
        candidates_for_earliest
        .groupby('cmdb_id', as_index=False)
        .agg(earliest_start_of_max_run=('earliest_start_of_longest_run','min'))
    )

    # max normalized severity among high points per cmdb_id
    max_sev_cmdb = (
        highs
        .groupby('cmdb_id', as_index=False)
        .agg(max_normalized_severity_among_high_points=('normalized_severity','max'))
    )

    # number of kpis with high runs per cmdb_id
    kpis_with_high_runs = (
        pair_longest[pair_longest['longest_run_length']>0]
        .groupby('cmdb_id', as_index=False)
        .agg(number_of_kpis_with_high_runs=('kpi_name','nunique'))
    )

    # Combine all per-cmdb aggregates
    per_cmdb_summary = per_cmdb_maxlen.merge(earliest_starts, on='cmdb_id', how='left')
    per_cmdb_summary = per_cmdb_summary.merge(max_sev_cmdb, on='cmdb_id', how='left')
    per_cmdb_summary = per_cmdb_summary.merge(kpis_with_high_runs, on='cmdb_id', how='left')

    # Fill NaNs for components with no kpis_with_high_runs (shouldn't happen since highs exists) and format
    per_cmdb_summary['number_of_kpis_with_high_runs'] = per_cmdb_summary['number_of_kpis_with_high_runs'].fillna(0).astype(int)
    per_cmdb_summary['max_normalized_severity_among_high_points'] = per_cmdb_summary['max_normalized_severity_among_high_points'].fillna(0).round(6)

    # Format earliest timestamp
    per_cmdb_summary['earliest_start_of_max_run'] = per_cmdb_summary['earliest_start_of_max_run'].dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')

# Sort by max_longest_high_run_length desc, then by max_normalized_severity desc, take top 10
top10_cmdb = per_cmdb_summary.sort_values(
    ['max_longest_high_run_length','max_normalized_severity_among_high_points'],
    ascending=[False, False]
).head(10).reset_index(drop=True)

# Final compact output: top10_cmdb (limited to 10 rows)
top10_cmdb

```
Out[8]:
```
```python
summary = (
    "Summary (incident window 2020-05-22 00:30–01:00 UTC+8):\n\n"
    "Top 10 components by longest consecutive 'high' run:\n"
    "1) os_007: longest high run = 31 minutes, start=2020-05-22 00:30:00, max_norm_sev=78.988002, kpis_with_high_runs=45\n"
    "2) os_020: longest high run = 31 minutes, start=2020-05-22 00:30:00, max_norm_sev=1.110342, kpis_with_high_runs=44\n"
    "3) docker_008: longest high run = 31 minutes, start=2020-05-22 00:30:00, max_norm_sev=0.188119, kpis_with_high_runs=9\n"
    "4) db_009:  longest high run = 30 minutes, start=2020-05-22 00:30:00, max_norm_sev=99.000000, kpis_with_high_runs=42\n"
    "5) db_002:  longest high run = 30 minutes, start=2020-05-22 00:30:00, max_norm_sev=79.975610, kpis_with_high_runs=43\n"
    "6) db_011:  longest high run = 30 minutes, start=2020-05-22 00:30:00, max_norm_sev=78.858156, kpis_with_high_runs=40\n"
    "7) db_004:  longest high run = 30 minutes, start=2020-05-22 00:30:00, max_norm_sev=19.000000, kpis_with_high_runs=36\n"
    "8) os_012:  longest high run = 30 minutes, start=2020-05-22 00:30:00, max_norm_sev=18.671735, kpis_with_high_runs=46\n"
    "9) os_018:  longest high run = 30 minutes, start=2020-05-22 00:30:00, max_norm_sev=12.313096, kpis_with_high_runs=40\n"
    "10) db_003: longest high run = 30 minutes, start=2020-05-22 00:30:00, max_norm_sev=9.230754, kpis_with_high_runs=44\n\n"
    "Interpretation:\n"
    "- os_007, os_020 and docker_008 show the longest uninterrupted high periods (31 minutes).\n"
    "- db_009 and db_002 exhibit the largest peak severities (99.0 and ~79.98 respectively) even though their longest runs are 30 minutes.\n"
    "- Prioritize investigation on components that combine long runs with high peak severity (notably os_007 and the high-severity DBs db_009 and db_002).\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  max_longest_high_run_length     earliest_start_of_max_run  max_normalized_severity_among_high_points  number_of_kpis_with_high_runs
0      os_007                           31  2020-05-22 00:30:00 CST+0800                                  78.988002                             45
1      os_020                           31  2020-05-22 00:30:00 CST+0800                                   1.110342                             44
2  docker_008                           31  2020-05-22 00:30:00 CST+0800                                   0.188119                              9
3      db_009                           30  2020-05-22 00:30:00 CST+0800                                  99.000000                             42
4      db_002                           30  2020-05-22 00:30:00 CST+0800                                  79.975610                             43
5      db_011                           30  2020-05-22 00:30:00 CST+0800                                  78.858156                             40
6      db_004                           30  2020-05-22 00:30:00 CST+0800                                  19.000000                             36
7      os_012                           30  2020-05-22 00:30:00 CST+0800                                  18.671735                             46
8      os_018                           30  2020-05-22 00:30:00 CST+0800                                  12.313096                             40
9      db_003                           30  2020-05-22 00:30:00 CST+0800                                   9.230754                             44```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz

# Reuse in-memory anomalies DataFrame 'anoms'
try:
    anoms
except NameError:
    try:
        anomalies
        anoms = anomalies.copy()
    except NameError:
        raise RuntimeError("Anomalies DataFrame not found in memory. Please run prior steps first.")

# Ensure timezone is set to UTC+8
tz = pytz.timezone('Asia/Shanghai')
if anoms['ts'].dt.tz is None:
    anoms['ts'] = anoms['ts'].dt.tz_localize(tz)
else:
    anoms['ts'] = anoms['ts'].dt.tz_convert(tz)

# Filter for db_009 and only 'high' anomalies within the incident window (anoms already limited to the window)
db009_high = anoms[(anoms['cmdb_id'] == 'db_009') & (anoms['anomaly_type'] == 'high')].copy()

# Aggregate by kpi_name
db009_high_kpi_summary = (
    db009_high
    .groupby('kpi_name', as_index=False)
    .agg(
        count_of_high_anomalies=('anomaly_type', 'count'),
        earliest_high_timestamp=('ts', 'min'),
        latest_high_timestamp=('ts', 'max'),
        max_normalized_severity=('normalized_severity', 'max'),
        mean_normalized_severity=('normalized_severity', 'mean')
    )
)

# Format timestamps and numeric columns for compact display
if not db009_high_kpi_summary.empty:
    db009_high_kpi_summary['earliest_high_timestamp'] = db009_high_kpi_summary['earliest_high_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')
    db009_high_kpi_summary['latest_high_timestamp'] = db009_high_kpi_summary['latest_high_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')
    db009_high_kpi_summary['max_normalized_severity'] = db009_high_kpi_summary['max_normalized_severity'].round(6)
    db009_high_kpi_summary['mean_normalized_severity'] = db009_high_kpi_summary['mean_normalized_severity'].round(6)

# Sort by max_normalized_severity descending and return top 20 rows (compact)
db009_high_kpi_summary_top20 = db009_high_kpi_summary.sort_values('max_normalized_severity', ascending=False).head(20).reset_index(drop=True)

# Display the result variable (compact table)
db009_high_kpi_summary_top20

```
Out[9]:
```
```python
summary = (
    "db_009 — high-anomaly summary by KPI (window 2020-05-22 00:30–01:00 UTC+8):\n\n"
    "Top KPIs by max normalized severity:\n"
    "1) SeqRead_Per_Sec: count=2, earliest=2020-05-22 00:52:00, latest=2020-05-22 00:56:00, "
    "max_norm_sev=99.0, mean_norm_sev≈49.9524\n"
    "2) Redo_Per_Sec: count=2, earliest=2020-05-22 00:52:00, latest=2020-05-22 00:58:00, "
    "max_norm_sev≈13.8080, mean≈10.7797\n"
    "3) TPS_Per_Sec: count=3, earliest=2020-05-22 00:33:00, latest=2020-05-22 00:59:00, "
    "max_norm_sev≈0.7082, mean≈0.2481\n"
    "4) User_Commit: count=2, earliest=00:38, latest=00:58, max_norm_sev≈0.3011, mean≈0.1649\n"
    "5) LFParaWrite_Per_Sec: count=2, earliest=00:52, latest=00:58, max_norm_sev≈0.2129, mean≈0.1860\n"
    "... (other KPIs up to 20 returned; note DFParaWrite_Per_Sec shows many anomaly rows: count=30 but max_norm_sev=0.0)\n\n"
    "Notes and recommendations:\n"
    "- The most severe KPI is SeqRead_Per_Sec (very large spike at 00:52). This should be investigated first.\n"
    "- Redo_Per_Sec and TPS_Per_Sec are the next-priority KPIs by severity and frequency.\n"
    "- Some KPIs (e.g., DFParaWrite_Per_Sec) have many anomaly rows but zero normalized severity—these likely equal their thresholds and are lower priority.\n"
    "- Action: drill into SeqRead_Per_Sec and Redo_Per_Sec around their earliest breach timestamps (00:52 onwards) for root-cause analysis."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name  count_of_high_anomalies       earliest_high_timestamp         latest_high_timestamp  max_normalized_severity  mean_normalized_severity
0         SeqRead_Per_Sec                        2  2020-05-22 00:52:00 CST+0800  2020-05-22 00:56:00 CST+0800                99.000000                 49.952381
1            Redo_Per_Sec                        2  2020-05-22 00:52:00 CST+0800  2020-05-22 00:58:00 CST+0800                13.808042                 10.779726
2             TPS_Per_Sec                        3  2020-05-22 00:33:00 CST+0800  2020-05-22 00:59:00 CST+0800                 0.708234                  0.248149
3             User_Commit                        2  2020-05-22 00:38:00 CST+0800  2020-05-22 00:58:00 CST+0800                 0.301059                  0.164902
4     LFParaWrite_Per_Sec                        2  2020-05-22 00:52:00 CST+0800  2020-05-22 00:58:00 CST+0800                 0.212938                  0.185984
5           Login_Per_Sec                        7  2020-05-22 00:42:00 CST+0800  2020-05-22 00:59:00 CST+0800                 0.166667                  0.023810
6            Call_Per_Sec                        1  2020-05-22 00:33:00 CST+0800  2020-05-22 00:33:00 CST+0800                 0.165805                  0.165805
7            Exec_Per_Sec                        3  2020-05-22 00:55:00 CST+0800  2020-05-22 00:59:00 CST+0800                 0.165700                  0.112722
8            CPU_Used_Pct                        2  2020-05-22 00:49:00 CST+0800  2020-05-22 00:58:00 CST+0800                 0.067106                  0.033993
9             Sess_Active                        5  2020-05-22 00:38:00 CST+0800  2020-05-22 00:59:00 CST+0800                 0.050000                  0.010000
10            Session_pct                        2  2020-05-22 00:48:00 CST+0800  2020-05-22 00:58:00 CST+0800                 0.031534                  0.020844
11         LFSync_Per_Sec                        1  2020-05-22 00:35:00 CST+0800  2020-05-22 00:35:00 CST+0800                 0.027900                  0.027900
12                 DbTime                        2  2020-05-22 00:48:00 CST+0800  2020-05-22 00:58:00 CST+0800                 0.022472                  0.011236
13  Physical_Read_Per_Sec                        2  2020-05-22 00:38:00 CST+0800  2020-05-22 00:58:00 CST+0800                 0.016940                  0.008663
14          MEM_real_util                        2  2020-05-22 00:33:00 CST+0800  2020-05-22 00:52:00 CST+0800                 0.002155                  0.001078
15     Logic_Read_Per_Sec                        1  2020-05-22 00:38:00 CST+0800  2020-05-22 00:38:00 CST+0800                 0.001587                  0.001587
16               MEM_Used                        1  2020-05-22 00:57:00 CST+0800  2020-05-22 00:57:00 CST+0800                 0.000066                  0.000066
17           CPU_free_pct                        1  2020-05-22 00:57:00 CST+0800  2020-05-22 00:57:00 CST+0800                 0.000049                  0.000049
18        New_Tbs_Free_Gb                        1  2020-05-22 00:57:00 CST+0800  2020-05-22 00:57:00 CST+0800                 0.000002                  0.000002
19    DFParaWrite_Per_Sec                       30  2020-05-22 00:30:00 CST+0800  2020-05-22 00:59:00 CST+0800                 0.000000                  0.000000```
```